In [9]:
import torch
import time
import json
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from datetime import datetime
from pathlib import Path

### Model loading


In [10]:
start = time.time()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# float16 causes placeholder tensor issues on MPS; use float32 on MPS, float16 on CUDA only
dtype = torch.float16 if device.type == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    "jinaai/jina-embeddings-v5-text-small", trust_remote_code=True
)
model = AutoModel.from_pretrained(
    "jinaai/jina-embeddings-v5-text-small",
    trust_remote_code=True,
    torch_dtype=dtype,
).to(device)

print("Using device:", device)
print(f"Model loaded in {time.time() - start:.2f}s")
print(f"Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Using device: mps
Model loaded in 12.24s
Model params: 676.8M


### Embedding Function


In [11]:
def get_embeddings(texts: list[str], batch_size: int = 8) -> torch.Tensor:
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        mask = encoded["attention_mask"].unsqueeze(-1).float()
        pooled = (output.last_hidden_state * mask).sum(1) / mask.sum(1)

        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())

    return torch.cat(all_embeddings, dim=0)

### Load & Inspect Training Data


In [12]:
# Notebook lives in notebooks/ so ../data/raw points to the project data/raw folder
DATA_PATH = Path("../data/raw/train.csv")

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['source'].value_counts()}")
df = df.drop(columns=["id"])
df.head()

Shape: (1392522, 3)
Columns: ['source', 'id', 'text']

Label distribution:
source
human    1028146
ai        364376
Name: count, dtype: int64


,source,text
0,human,12 Years a Slave: An Analysis of the Film Essa...
1,human,20+ Social Media Post Ideas to Radically Simpl...
2,human,2022 Russian Invasion of Ukraine in Global Med...
3,human,533 U.S. 27 (2001) Kyllo v. United States: The...
4,human,A Charles Schwab Corporation Case Essay\n\nCha...


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1392522 entries, 0 to 1392521
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   source  1392522 non-null  object
 1   text    1392362 non-null  object
dtypes: object(2)
memory usage: 21.2+ MB


## Prepare Training Data

Sample a balanced subset (the full dataset is very large) and map labels to binary: `0 = human`, `1 = AI`.


In [14]:
from sklearn.model_selection import train_test_split

MAX_PER_CLASS = 50000  # cap per class to keep embedding time reasonable

# Binary label: human=0, AI=1
df["label"] = df["source"].apply(lambda x: 0 if str(x).lower() == "human" else 1)

# Drop nulls and ensure strings
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

# Balanced sample (capped)
n = min(MAX_PER_CLASS, df["label"].value_counts().min())
human_df = df[df["label"] == 0].sample(n=n, random_state=42)
ai_df = df[df["label"] == 1].sample(n=n, random_state=42)
balanced_df = (
    pd.concat([human_df, ai_df]).sample(frac=1, random_state=42).reset_index(drop=True)
)

all_texts = balanced_df["text"].tolist()
all_labels = balanced_df["label"].tolist()

# 50 / 50 train-test split
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    all_texts, all_labels, test_size=0.25, random_state=42, stratify=all_labels
)

print(f"Total balanced samples : {len(all_texts)}")
print(f"Train : {len(X_train_texts)}  |  Test : {len(X_test_texts)}")
print(f"Train —> Human: {y_train.count(0)}  AI: {y_train.count(1)}")
print(f"Test  —> Human: {y_test.count(0)}   AI: {y_test.count(1)}")

Total balanced samples : 100000
Train : 75000  |  Test : 25000
Train —> Human: 37500  AI: 37500
Test  —> Human: 12500   AI: 12500


In [15]:
df.isna().sum()

source    0
text      0
label     0
dtype: int64

## Generate Embeddings


In [ ]:
t0 = time.time()

print("Embedding train set...")
X_train = get_embeddings(X_train_texts)
print(f"  Train embeddings: {X_train.shape}")

print("Embedding test set...")
X_test = get_embeddings(X_test_texts)
print(f"  Test  embeddings: {X_test.shape}")

print(f"\nDone in {time.time() - t0:.1f}s")

Embedding train set...


In [ ]:
hellomalekum

## Training models


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Naïve Bayes": GaussianNB(),
}

trained_models = {}
for name, m in models.items():
    print(f"Training {name}...")
    m.fit(X_train, y_train)
    trained_models[name] = m

print("\nAll models trained!")

In [ ]:
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset

# class TextClassifierNN(nn.Module):

#     def __init__(self, input_dim: int):
#         super().__init__()

#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.BatchNorm1d(256),
#             nn.ReLU(),
#             nn.Dropout(0.3),

#             nn.Linear(256, 128),
#             nn.BatchNorm1d(128),
#             nn.ReLU(),

#             nn.Dropout(0.2),
#             nn.Linear(64, 2),
#             nn.ReLU(),

#             nn.Linear(64, 2)

#         )


#     def forward(self, x):
#         return self.net(x)

#     def train_torch(X_train, y_train, epochs = 20, batch_size= 64, lr= 1e-3):
#         input_dim = X_train.shape[1]
#         nn_model = TextClassifierNN(input_dim).to(device)

#         # data loading part
#         X_t = torch.tensor(X_train, dtype= torch.float32)
#         y_t = torch.tensor(y_train, dtype = torch.long)
#         loader = DataLoader(TensorDataset(X_t, y_t), batch_size= batch_size, shuffle= True)

#         # optimization part
#         optimizer = torch.optim.Adam(nn_model.parameters(), lr= lr, weight_decay= 1e-4)
#         criteration = nn.CrossEntropyLoss()
#         scheduler = torch.optim.lr_scheduler.StepLR(optimizer= optimizer, step_size =5, gamma = 0.5)

#         # training

#         nn_model.train()
#         for epoch in range(epochs):
#             total_loss = 0
#             correct = 0
#             for xb, yb in loader:
#                 xb, yb = xb.to(device), yb.to(device)
#                 optimizer.zero_grad()
#                 out

## Evaluation — Metrics Table


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

results = {}
for name, m in trained_models.items():
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1]
    results[name] = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "ROC AUC": round(roc_auc_score(y_test, y_prob), 4),
    }

results_df = pd.DataFrame(results).T.sort_values("F1", ascending=False)
print(results_df.to_string())
results_df

## ROC AUC Curves & Model Comparison


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

# ── Figure 1: ROC AUC curves ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for (name, m), color in zip(trained_models.items(), colors):
    y_prob = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = results[name]["ROC AUC"]
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.3f})", color=color, lw=2)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC AUC Curves — All Models", fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# ── Figure 2: Metrics bar comparison ─────────────────────────────────────────
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]
model_names = list(results.keys())

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5))
for ax, metric in zip(axes, metrics):
    values = [results[name][metric] for name in model_names]
    bars = ax.bar(model_names, values, color=colors)
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=20, ha="right", fontsize=8)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}",
            ha="center",
            fontsize=8,
        )

plt.suptitle("Model Comparison — All Metrics", fontsize=14)
plt.tight_layout()
plt.show()

## Best Model Selection


In [ ]:
best_name = results_df["F1"].idxmax()
best_clf = trained_models[best_name]

print(f"Best model  : {best_name}")
print(f"  Accuracy  : {results[best_name]['Accuracy']:.4f}")
print(f"  Precision : {results[best_name]['Precision']:.4f}")
print(f"  Recall    : {results[best_name]['Recall']:.4f}")
print(f"  F1        : {results[best_name]['F1']:.4f}")
print(f"  ROC AUC   : {results[best_name]['ROC AUC']:.4f}")
print(f"\n→ Using '{best_name}' for the interactive detector.")

## Predict Function


In [ ]:
def predict(text: str):
    emb = get_embeddings([text])
    prob = best_clf.predict_proba(emb)[0]  # uses the best model
    ai_prob = prob[1] * 100
    human_prob = prob[0] * 100

    label = "AI-written" if prob[1] >= 0.5 else "Human-written"

    bar_len = 30
    ai_bar = "█" * int(ai_prob / 100 * bar_len)
    human_bar = "█" * int(human_prob / 100 * bar_len)

    print("\n")
    if label == "AI-written":
        print("Verdict: AI generated")
    else:
        print("Verdict: Human written")

    print(f"Confidence: {max(ai_prob, human_prob):.1f} %")
    print(f"\nAI probability:    {ai_bar:<30} {ai_prob:.1f} %")
    print(f"\nHuman probability: {human_bar:<30} {human_prob:.1f} %\n\n")

    return ai_prob, human_prob

In [ ]:
LOG_PATH = Path("new_predictions_log.json")


def save_result(record: dict):
    """Append prediction record to JSON file safely."""
    if LOG_PATH.exists():
        with open(LOG_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = []

    data.append(record)

    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

## Interactive Detector


In [ ]:
print("AI text detector\n")
print("Type or Paste text and press Enter !")
print("Type 'quit' or 'exit' to stop !\n")

while True:
    try:
        text = input("Enter text: ").strip()

        if not text:
            print("Please enter some text")
            continue
        if text.lower() in ("quit", "exit", "q"):
            print("Goodbye gentleman\n")
            break
        if len(text.split()) < 5:
            print("Please enter at least 5 words for reliable results!")
            continue

        start_time = time.perf_counter()
        ai_prob, human_prob = predict(text)
        elapsed = time.perf_counter() - start_time

        print(f"AI probability: {ai_prob:.4f}")
        print(f"Human probability: {human_prob:.4f}")
        print(f"Time taken: {elapsed:.3f} sec")

        record = {
            "timestamp": datetime.utcnow().isoformat(),
            "text": text,
            "ai_probability": float(ai_prob),
            "human_probability": float(human_prob),
            "inference_time_sec": round(elapsed, 4),
        }

        save_result(record)
        print("Saved to predictions_log.json\n")

    except KeyboardInterrupt:
        print("\nGoodbye dont try to use AI otherwise u get dumb")
        break

AI text detector

Type or Paste text and press Enter !
Type 'quit' or 'exit' to stop !



Verdict: Human written
Confidence: 87.1 %

AI probability:    ███                            12.88 %

Human probability: ██████████████████████████     87.12 %


AI probability: 12.8800
Human probability: 87.1200
Time taken: 0.679 sec
Saved to predictions_log.json



/var/folders/sx/s2q0vz2d1mv8hq3w30ss93l40000gn/T/ipykernel_90134/2331035508.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


Goodbye gentleman

